<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c02-header.png" alt="Nextia Learning · Python for Practical AI Engineering" width="100%">

# Solution: Notebook discipline

**[C02-M06-L01 · Notebook discipline](https://learning.nextia-ai.com/courses/python/m06/notebook-discipline/)** · C02, Python for Practical AI Engineering · Module 6, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** find the hidden state that makes a notebook give a different result each time, and make a notebook that gives the same result from a fresh start.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup, to download the course's ticket file (1 KB). No account and no packages beyond Python itself. |
| **You should know** | Modules and imports, from [Modules and imports](https://learning.nextia-ai.com/courses/python/m03/modules-and-imports/). |
| **You do not need** | Your local project. |
| **Tested** | 2026-10-07, Python 3.14.6, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook makes each kind of hidden state happen on purpose, so that you can see it. The [lesson page](https://learning.nextia-ai.com/courses/python/m06/notebook-discipline/) has the full explanations, the diagram and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells one at a time, from the top. Press **Shift+Enter** to run a cell and move to the next one. Read the text cell before each code cell.
2. At a **Predict** note, write your prediction down before you run the cell.
3. Some steps ask you to change a cell, delete a cell, or restart the notebook. Do exactly what the step says.
4. At an exercise, write your answer in the cell, run it, then run the check cell below it. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C02/M06-L01-notebook-discipline/notebook-discipline-solution.ipynb).

## Setup: make the project folder

Run the next cell. It makes a folder `ticket-cleaner`, laid out like your project from Module 1, and works in it. It is empty except for the `data` folder, as at the end of [Files and terminal](https://learning.nextia-ai.com/courses/python/m01/files-and-terminal/#guided-practice-make-your-project-folder). It needs no internet connection.

It also downloads the course's ticket files (`tickets.csv`, under 4 KB together) into `data/`, and checks each file's checksum, as you did with `curl` in [Reading and writing files](https://learning.nextia-ai.com/courses/python/m04/reading-and-writing-files/#get-the-data-file). This part needs an internet connection and no account. If it shows `Could not download`, check your connection and run it again (in Kaggle, turn on **Internet** in the notebook settings). If it shows `wrong checksum`, delete the folder `ticket-cleaner` (in Colab: the **Files** panel on the left), then run it again. The data is made up for this course; it has no real people.

You should see `Ready: ticket-cleaner, 1 file`. If you run the cell again, it puts every file back as at the start of this lesson: use this to start again.

In [ ]:
# Setup: make the ticket-cleaner folder as at the start of this lesson. Download the data files.
import hashlib, os, shutil, subprocess, sys, urllib.request
from pathlib import Path

if Path.cwd().name == "ticket-cleaner":  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-cleaner").resolve()
for item in PROJECT.glob("*"):  # start again: remove everything except the data files
    if item.name != "data":
        shutil.rmtree(item) if item.is_dir() else item.unlink()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C02/"
DATA = {
    "tickets.csv": "82e3fb4fa6d94a47b9a7b05d940a088596363e4dd11e89bf82ed486bd2325a1a",
}

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in DATA.items():
    path = PROJECT / "data" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-cleaner and run again.")


os.chdir(PROJECT)  # work in the project folder, as in the lesson
sys.path.insert(0, str(PROJECT))
sys.dont_write_bytecode = True  # always read the newest version of each file
count = sum(1 for path in PROJECT.rglob("*") if path.is_file())
print("Ready: ticket-cleaner" + (f", {count} file{'s' if count > 1 else ''}" if count else ""))

## Setup: the helpers

The next cell defines small functions that this notebook uses. You do not need to read them, but you can.

- `check("""assert ...""")` runs the `assert` lines of a check cell. It prints `Check passed.`, or `Not yet:` and what is different. It does not show the answer.

Run the cell. It prints nothing.

In [ ]:
# Helpers: run the project, and check an answer without showing it.
def check(asserts):
    """Run the assert lines of a check. Say what is different, never the answer."""
    try:
        exec(asserts, globals())
    except AssertionError as error:
        print(f"Not yet: {error}")
    except NameError as error:
        print(f"Not yet: {error}. Run your cell above first.")
    except Exception as error:
        print(f"Not yet: the check stopped with {type(error).__name__}: {error}")
    else:
        print("Check passed.")

Run the next cell. It reads the ticket file into `rows`, a list of dictionaries. You should see `15 rows`.

In [ ]:
import csv

with open("data/tickets.csv", encoding="utf-8-sig", newline="") as file:
    rows = list(csv.DictReader(file))
print(len(rows), "rows")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## The idea: the kernel remembers, the page does not

A notebook has two parts. The **page** shows your cells. The **kernel** is the Python program that runs them, and it keeps every variable in memory until it restarts.

The kernel does not care about the order of the cells on the page. It runs a cell when you run it, with the variables that exist **at that moment**. The number in `[ ]` next to a cell shows when it ran: `[5]` ran after `[4]`, wherever it is on the page.

So the page can show one story while the kernel holds a different one. The next cells show how.

Run the next cell. It chooses which status to count.

In [ ]:
wanted_status = "open"

Run the next cell. It counts the rows with the wanted status, after cleaning the text.

You should see `open 9`.

In [ ]:
selected = [row for row in rows if row["status"].strip().lower() == wanted_status]
print(wanted_status, len(selected))

## Predict, then run

1. Go back to the cell `wanted_status = "open"`. Change it to `wanted_status = "closed"`, and run **only that cell**.
2. Do not run the count cell again.
3. **Predict:** what does the next cell print? Write your prediction down first.
4. Run the next cell.

In [ ]:
print(wanted_status, len(selected))

It prints `closed 9`. But only 3 tickets are closed. `wanted_status` is new, and `selected` is still the result for `"open"`, because the cell that makes `selected` did not run again. The page shows `"closed"` everywhere, so a reader believes that 9 tickets are closed.

This is **hidden state**: values in the kernel that the page does not explain. Change `wanted_status` back to `"open"` before you continue.

## A deleted cell that still counts

Run the next cell. Then **delete it**: select the cell, and use the delete command of your notebook (for example **Edit › Delete Cells** in JupyterLab, or the bin icon in Colab).

In [ ]:
priority_limit = 2

Run the next cell. It still works, although the cell that made `priority_limit` is gone: the kernel remembers it.

In [ ]:
urgent_enough = [row for row in rows if row["priority"].strip() in {"1", str(priority_limit)}]
print(len(urgent_enough), "rows with priority 1 or", priority_limit)

## Failure case: Restart and run all

Now give the notebook a fresh start, as a colleague who opens it would have:

- **JupyterLab:** Kernel › Restart Kernel and Run All Cells.
- **Colab:** Runtime › Restart session and run all.
- **Kaggle:** Run › Restart & clear cell outputs, then Run › Run all.

If you deleted the `priority_limit = 2` cell, the run stops at the cell above with:

```
NameError: name 'priority_limit' is not defined
```

The notebook worked only because of something that is no longer on the page. **Fix:** put the value back in a cell **above** the cell that uses it, then restart and run all again, until it runs from top to bottom without an error. Do this before you share a notebook, every time.

## Move reusable code out of the notebook

A function that you want to keep, test and use in a program belongs in a module, not in a cell. The cell magic `%%writefile` saves a cell as a file. Run the next cell: it makes the module `ticket_rules.py` in the notebook's working folder.

In [ ]:
%%writefile ticket_rules.py
RULES_VERSION = 1


def clean_status(value: str) -> str:
    """Return the status with no outer spaces, in small letters."""
    return value.strip().lower()

Run the next cell. It imports the module and uses it. You should see `1 open`.

In [ ]:
import ticket_rules

print(ticket_rules.RULES_VERSION, ticket_rules.clean_status("  Open "))

## Change one thing: a new version of the module

Run the next cell. It saves version 2 of the module, which adds a function. Then run the cell after it, which imports the module again.

**Predict:** which version number does it print?

In [ ]:
%%writefile ticket_rules.py
RULES_VERSION = 2


def clean_status(value: str) -> str:
    """Return the status with no outer spaces, in small letters."""
    return value.strip().lower()


def is_closed(value: str) -> bool:
    """Return True if the cleaned status is closed."""
    return clean_status(value) == "closed"

In [ ]:
import ticket_rules

print(ticket_rules.RULES_VERSION, hasattr(ticket_rules, "is_closed"))

It prints `1 False`: version 1, without the new function. Python imports a module only once per kernel. The second `import` uses the copy in memory, so the kernel still has version 1, although the file has version 2. This is hidden state too.

Two fixes: restart the kernel (then every import reads the files again), or reload the one module, as in the next cell. You should see `2 True`.

In [ ]:
import importlib

importlib.reload(ticket_rules)
print(ticket_rules.RULES_VERSION, hasattr(ticket_rules, "is_closed"))

## Check yourself

Each exercise ends with a check cell. If your answer is correct, it prints `Check passed.` If not, it prints `Not yet:` and what to check.

**Exercise 1.** In the next cell, count the rows with the status `pending` (after cleaning) into `pending_count`.

In [ ]:
wanted_status = "pending"
pending_count = len([row for row in rows if row["status"].strip().lower() == wanted_status])

In [ ]:
check("""
assert isinstance(pending_count, int), "pending_count must be a number. Use len() on a list of the matching rows."
assert pending_count == len([r for r in rows if r["status"].strip().lower() == "pending"]), f"pending_count is {pending_count}. Count the rows whose cleaned status is 'pending'."
""")

**Exercise 2.** Add a function `is_known_status(value)` to the module. It returns `True` if the cleaned value is `open`, `pending` or `closed`. Save the module with the next cell. The cell after it reloads the module and checks your function: 14 of the 15 rows have a known status (row 7 has `opne`).

In [ ]:
%%writefile ticket_rules.py
RULES_VERSION = 3
STATUSES = {"open", "pending", "closed"}


def clean_status(value: str) -> str:
    """Return the status with no outer spaces, in small letters."""
    return value.strip().lower()


def is_known_status(value: str) -> bool:
    """Return True if the cleaned value is one of the allowed statuses."""
    return clean_status(value) in STATUSES

In [ ]:
importlib.reload(ticket_rules)

check("""
assert ticket_rules.RULES_VERSION == 3, "Run the %%writefile cell above first, then this cell."
assert hasattr(ticket_rules, "is_known_status"), "The module has no function is_known_status yet."
known = [row for row in rows if ticket_rules.is_known_status(row["status"])]
assert len(known) == 14, f"{len(known)} rows have a known status. There must be 14."
assert not ticket_rules.is_known_status("opne"), "'opne' is not a known status."
""")

## Recap

- The kernel keeps every variable until it restarts. The order in which you ran the cells is what counts, not their order on the page.
- A changed cell that you did not run again, and a deleted cell, both leave **hidden state**.
- Before you share a notebook, **restart and run all**. It must run from top to bottom without an error.
- Keep exploration in the notebook. Move code that you want to keep into a module. After you change a module, reload it or restart the kernel.

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/python/m06/notebook-discipline/). They count toward your certificate when you are signed in and enrolled.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Command-line interface](https://learning.nextia-ai.com/courses/python/m06/command-line-interface/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/python/m06/notebook-discipline/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets in this course are made up; they have no real people.